# Imports

In [1]:
import pandas as pd
from datetime import timedelta

# Data load and parameter set

In [2]:
# === PARAMETERS ===
RAIN_ACCUM_HOURS = 24
RAIN_THRESHOLD_MM = 0.1
STEP_HOURS = 12
MAX_STEPS = 100

# === File paths ===
gv_path = r"D:\Development\RESEARCH\Gazelle_Valley\Data\Precipiteion_Data\gazelle_valley\20230921_20241117_processed.csv"
ziv_path = r"D:\Development\RESEARCH\Gazelle_Valley\Data\Precipiteion_Data\ziv\20230921_20241117_processed.csv"
ram_path = r"D:\Development\RESEARCH\Gazelle_Valley\Data\Precipiteion_Data\givat_ram\20231004_20250406_processed.csv"

# === Load rain data ===
gv_rain = pd.read_csv(gv_path, parse_dates=['date_time'])
ziv_rain = pd.read_csv(ziv_path, parse_dates=['date_time'])
ram_rain = pd.read_csv(ram_path, parse_dates=['date_time'])

rain_stations = [gv_rain, ziv_rain, ram_rain]
station_names = ['gazelle_valley', 'ziv', 'givat_ram']

for rain_df in rain_stations:
    rain_df.set_index('date_time', inplace=True)
    rain_df.sort_index(inplace=True)
    rain_df.index = rain_df.index.tz_localize(None)

# === Load and clean flow events ===
flow_events = pd.read_csv(r"D:\Development\RESEARCH\Gazelle_Valley\Data\discharge\flow_events_processed.csv")
flow_events['flow_start'] = pd.to_datetime(flow_events['flow_start_date'] + ' ' + flow_events['flow_start_time'], dayfirst=True, errors='coerce')
flow_events['flow_end'] = pd.to_datetime(flow_events['flow_end_date'] + ' ' + flow_events['flow_end_time'], dayfirst=True, errors='coerce')
flow_events.drop(columns=['flow_start_date', 'flow_start_time', 'flow_end_date', 'flow_end_time'], inplace=True)
flow_events.dropna(subset=['flow_start', 'flow_end'], inplace=True)

# Combine flow peak date + time
flow_events['flow_peak_datetime'] = pd.to_datetime(
    flow_events['flow_peak_date'].astype(str) + ' ' + flow_events['flow_peak_time'].astype(str),
    dayfirst=True, errors='coerce'
)
flow_events.drop(columns=['flow_peak_date', 'flow_peak_time'], inplace=True)

# Functions

In [3]:
# === Find rain boundary (start or end of rainfall event) ===
def find_rain_boundary(rain_dfs, flow_time, direction):
    delta = timedelta(hours=RAIN_ACCUM_HOURS)
    step = timedelta(hours=STEP_HOURS)
    current_times = [flow_time - step if direction == 'backward' else flow_time + step for _ in rain_dfs]
    steps = 0

    while steps < MAX_STEPS:
        all_below = []
        for i, rain_df in enumerate(rain_dfs):
            t = current_times[i]
            if direction == 'backward':
                rain_amount = rain_df.loc[t - delta:t]['rain_mm'].sum()
            else:
                rain_amount = rain_df.loc[t:t + delta]['rain_mm'].sum()

            if rain_amount < RAIN_THRESHOLD_MM:
                all_below.append(True)
            else:
                current_times[i] = current_times[i] - step if direction == 'backward' else current_times[i] + step
                all_below.append(False)

        if all(all_below):
            return max(current_times) if direction == 'backward' else min(current_times)
        steps += 1

    return None


# === Summarize rainfall statistics per station ===
def summarize_rain_events(rain_event_df, rain_df):
    total_mm, max_intensity, max_datetime = [], [], []
    for _, row in rain_event_df.iterrows():
        start, end = row['rain_event_start'], row['rain_event_end']
        event_rain = rain_df.loc[start:end]
        total = event_rain['rain_mm'].sum()
        max_val = event_rain['rain_intensity_mm_hr'].max()
        if pd.notna(max_val) and not event_rain.empty:
            max_row = event_rain[event_rain['rain_intensity_mm_hr'] == max_val].iloc[0]
            max_dt = max_row.name
        else:
            max_dt = pd.NaT
        total_mm.append(total)
        max_intensity.append(max_val)
        max_datetime.append(max_dt)
    return total_mm, max_intensity, max_datetime


In [4]:
def assign_event_id(df, date_col='flow_start'):
    """
    Returns a new Series of event IDs based on rainy season logic:
    - Each rainy season starts in September and ends in August the next year.
    - Event ID is [season_index][event_number], e.g. 101, 102, ..., 201, 202...

    Parameters:
    - df (pd.DataFrame): The DataFrame containing the event dates.
    - date_col (str): The column name to base the season on (default: 'flow_start').

    Returns:
    - pd.Series: A Series of event IDs (Int64) with same index as df.
    """
    temp = df[[date_col]].copy()
    temp['season_year'] = temp[date_col].apply(lambda dt: dt.year if dt.month >= 9 else dt.year - 1)
    temp['season_index'] = temp['season_year'] - temp['season_year'].min() + 1
    temp['event_number'] = temp.groupby('season_year').cumcount() + 1
    event_ids = temp['season_index'] * 100 + temp['event_number']
    return event_ids.astype('Int64')


In [5]:
def filter_events_by_total_rain(df, threshold_mm, station_names=None):
    """
    Filters rainflow_events_df based on total rainfall threshold.
    Keeps events where at least one station has total_mm >= threshold.
    Ignores events where all values are below threshold or NaN.
    
    Parameters:
    - df (pd.DataFrame): MultiIndex DataFrame of events
    - threshold_mm (float): Rainfall threshold in mm
    - station_names (list of str): Names of the stations (e.g. ['gazelle_valley', 'ziv', 'givat_ram'])

    Returns:
    - pd.DataFrame: Filtered DataFrame
    """
    if station_names is None:
        station_names = ['gazelle_valley', 'ziv', 'givat_ram']
        
    mask = pd.Series(False, index=df.index)
    
    for name in station_names:
        rain = df[(name, 'total_mm')]
        valid_and_high = rain.notna() & (rain >= threshold_mm)
        mask = mask | valid_and_high

    return df[mask]


# Main

In [6]:

# === Determine rain event start and end for each flow event ===
rain_start_list, rain_end_list = [], []

for _, row in flow_events.iterrows():
    start_dt = row['flow_start']
    end_dt = row['flow_end']
    rain_start = find_rain_boundary(rain_stations, start_dt, direction='backward')
    rain_end = find_rain_boundary(rain_stations, end_dt, direction='forward')
    rain_start_list.append(rain_start)
    rain_end_list.append(rain_end)

flow_events['rain_event_start'] = rain_start_list
flow_events['rain_event_end'] = rain_end_list


# === Build MultiIndex dataframe ===
multi_df = {}

# Add non-time flow data
for col in flow_events.columns.difference(['flow_start', 'flow_end', 'rain_event_start', 'rain_event_end']):
    multi_df[('discharge', col)] = flow_events[col]

# Add rain timing
multi_df[('rain', 'rain_event_start')] = flow_events['rain_event_start']
multi_df[('rain', 'rain_event_end')] = flow_events['rain_event_end']

# Add stats per station
for rain_df, name in zip(rain_stations, station_names):
    total_mm, max_intensity, max_datetime = summarize_rain_events(flow_events, rain_df)
    multi_df[(name, 'total_mm')] = total_mm
    multi_df[(name, 'max_intensity_mm_hr')] = max_intensity
    multi_df[(name, 'max_intensity_datetime')] = max_datetime

# Final dataframe
raw_rainflow_events_df = pd.DataFrame(multi_df)
raw_rainflow_events_df.columns = pd.MultiIndex.from_tuples(raw_rainflow_events_df.columns)

# Set event idx by hydrological seasone
raw_rainflow_events_df['season_index'] = assign_event_id(flow_events)
raw_rainflow_events_df.set_index('season_index', inplace=True)
raw_rainflow_events_df

discharge                                      \
              flow_peak_datetime flow_peak_m3_s flow_total_volume_m3   
season_index                                                           
101          2023-11-01 11:50:00       0.617218          1773.301946   
102          2023-11-14 14:40:00       0.063550            39.911987   
103          2023-11-20 00:55:00       0.394052          7530.163635   
104          2023-11-27 11:30:00       0.218358          2319.086856   
105          2023-12-05 15:50:00       0.166977           545.809527   
106          2023-12-13 04:00:00       0.247912          1022.017072   
107          2023-12-24 04:05:00       0.306212          5235.336201   
108          2024-01-02 07:35:00       0.283269         10427.108320   
109          2024-01-12 02:50:00       0.292369         10366.953360   
110          2024-01-18 13:40:00       0.239335           472.598758   
111          2024-01-26 15:15:00       0.399168         17738.155380   
112          2024-02-02 05:45:00       0.198052          5783.383577   
113          2024-02-14 18:20:00       0.038764            60.869573   
114          2024-02-19 03:35:00       0.265381          4853.161293   
115          2024-02-27 10:35:00       0.155777          1065.196421   
116          2024-03-03 07:05:00       0.023595             7.537237   
117          2024-03-12 19:25:00       0.148450           644.989093   
118          2024-03-19 07:00:00       0.889764         14551.698730   
119          2024-04-01 23:45:00       0.033586            81.763799   
120          2024-05-01 12:05:00       0.014092             5.752664   
121          2024-05-06 08:30:00       0.159483           187.342221   
122          2024-07-03 12:35:00       0.063550           280.542480   
123          2024-07-18 07:05:00       0.016422            92.255190   
201          2024-11-02 15:20:00       0.163216           130.210958   
202          2024-11-26 04:25:00       0.218358           640.819036   
203          2024-12-13 15:45:00       0.014092             5.100218   
204          2024-12-15 23:10:00       0.038764           103.170565   
205          2024-12-20 12:40:00       0.178424           327.618529   
206          2024-12-31 12:15:00       0.599202          8685.899561   
207          2025-01-11 05:05:00       0.033586            35.699259   
208          2025-02-05 21:30:00       0.896697          8231.422792   
209          2025-02-09 23:25:00       0.063550            89.782286   
210          2025-02-20 10:45:00       0.889764          1620.773314   
211          2025-03-07 06:40:00       0.182295          1090.477842   
212          2025-03-20 18:45:00       0.301572          2665.734221   

                            rain                     gazelle_valley  \
                rain_event_start      rain_event_end       total_mm   
season_index                                                          
101          2023-10-31 22:20:00 2023-11-04 10:40:00           10.7   
102          2023-11-14 02:30:00 2023-11-15 03:05:00            1.1   
103          2023-11-19 00:20:00 2023-11-22 02:15:00           32.8   
104          2023-11-26 14:05:00 2023-11-28 04:10:00           20.5   
105          2023-12-05 02:20:00 2023-12-06 22:40:00           11.7   
106          2023-12-12 14:35:00 2023-12-14 01:30:00           10.4   
107          2023-12-21 15:55:00 2023-12-25 02:55:00           29.9   
108          2024-01-01 12:40:00 2024-01-04 07:20:00            0.8   
109          2024-01-10 09:50:00 2024-01-15 21:40:00           52.9   
110          2024-01-18 01:35:00 2024-01-19 02:45:00            0.2   
111          2024-01-22 23:20:00 2024-01-31 10:50:00           74.9   
112          2024-02-01 11:55:00 2024-02-07 11:25:00           27.5   
113          2024-02-13 10:50:00 2024-02-16 06:50:00            3.8   
114          2024-02-17 08:20:00 2024-02-24 12:00:00           18.4   
115          2024-02-26 22:30:00 2024-02-29 10:50:00            2.8   
116     

## Filter event by total rainfall

In [7]:
total_rain_threshold_mm=1 # units: mm
rainflow_events_df = filter_events_by_total_rain(raw_rainflow_events_df, total_rain_threshold_mm)
print(f"Filtered out {len(raw_rainflow_events_df) - len(rainflow_events_df)} events below threshold.")

Filtered out 8 events below threshold.


In [8]:
rainflow_events_df

discharge                                      \
              flow_peak_datetime flow_peak_m3_s flow_total_volume_m3   
season_index                                                           
101          2023-11-01 11:50:00       0.617218          1773.301946   
102          2023-11-14 14:40:00       0.063550            39.911987   
103          2023-11-20 00:55:00       0.394052          7530.163635   
104          2023-11-27 11:30:00       0.218358          2319.086856   
105          2023-12-05 15:50:00       0.166977           545.809527   
106          2023-12-13 04:00:00       0.247912          1022.017072   
107          2023-12-24 04:05:00       0.306212          5235.336201   
108          2024-01-02 07:35:00       0.283269         10427.108320   
109          2024-01-12 02:50:00       0.292369         10366.953360   
111          2024-01-26 15:15:00       0.399168         17738.155380   
112          2024-02-02 05:45:00       0.198052          5783.383577   
113          2024-02-14 18:20:00       0.038764            60.869573   
114          2024-02-19 03:35:00       0.265381          4853.161293   
115          2024-02-27 10:35:00       0.155777          1065.196421   
116          2024-03-03 07:05:00       0.023595             7.537237   
118          2024-03-19 07:00:00       0.889764         14551.698730   
121          2024-05-06 08:30:00       0.159483           187.342221   
201          2024-11-02 15:20:00       0.163216           130.210958   
202          2024-11-26 04:25:00       0.218358           640.819036   
205          2024-12-20 12:40:00       0.178424           327.618529   
206          2024-12-31 12:15:00       0.599202          8685.899561   
207          2025-01-11 05:05:00       0.033586            35.699259   
208          2025-02-05 21:30:00       0.896697          8231.422792   
209          2025-02-09 23:25:00       0.063550            89.782286   
210          2025-02-20 10:45:00       0.889764          1620.773314   
211          2025-03-07 06:40:00       0.182295          1090.477842   
212          2025-03-20 18:45:00       0.301572          2665.734221   

                            rain                     gazelle_valley  \
                rain_event_start      rain_event_end       total_mm   
season_index                                                          
101          2023-10-31 22:20:00 2023-11-04 10:40:00           10.7   
102          2023-11-14 02:30:00 2023-11-15 03:05:00            1.1   
103          2023-11-19 00:20:00 2023-11-22 02:15:00           32.8   
104          2023-11-26 14:05:00 2023-11-28 04:10:00           20.5   
105          2023-12-05 02:20:00 2023-12-06 22:40:00           11.7   
106          2023-12-12 14:35:00 2023-12-14 01:30:00           10.4   
107          2023-12-21 15:55:00 2023-12-25 02:55:00           29.9   
108          2024-01-01 12:40:00 2024-01-04 07:20:00            0.8   
109          2024-01-10 09:50:00 2024-01-15 21:40:00           52.9   
111          2024-01-22 23:20:00 2024-01-31 10:50:00           74.9   
112          2024-02-01 11:55:00 2024-02-07 11:25:00           27.5   
113          2024-02-13 10:50:00 2024-02-16 06:50:00            3.8   
114          2024-02-17 08:20:00 2024-02-24 12:00:00           18.4   
115          2024-02-26 22:30:00 2024-02-29 10:50:00            2.8   
116          2024-03-02 19:00:00 2024-03-04 07:20:00            3.9   
118          2024-03-18 00:50:00 2024-03-20 12:00:00           37.8   
121          2024-05-05 16:20:00 2024-05-06 21:35:00            4.7   
201          2024-11-02 03:10:00 2024-11-04 15:45:00            0.7   
202          2024-11-24 11:55:00 2024-11-26 17:05:00            0.0   
205          2024-12-19 23:55:00 2024-12-21 01:20:00            0.0   
206          2024-12-27 22:20:00 2025-01-01 02:10:00            0.0   
207          2025-01-10 16:30:00 2025-01-12 03:45:00            0.0   
208          2025-02-05 00:05:00 2025-02-07 23:30:00            0.0   
209          202